# FPR Experiment for Naive and Bonferroni p-values

This notebook estimates null false positive rates (FPRs) for two baselines after running the Adaptive CoRT pipeline on each dataset and drawing one random selected target feature per run.

- `Naive`: ignore the selection event and compute the usual two-sided Gaussian p-value from the data-dependent test statistic.
- `Bonferroni`: apply a Bonferroni correction with family size $2^p$, computed in log-space so the adjustment remains stable even when $2^p$ is extremely large.

In [ ]:
# # For kaggle:
# !git clone https://github.com/maTiahK/CORT_SI.git
# %cd /kaggle/working/CORT_SI

# !python -m pip install -q --upgrade pip
# !python -m pip install -q -e .


# from cort_si import gen_data
# print("CORT_SI import OK")

In [ ]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
REPO_ROOT = next((candidate for candidate in candidates if (candidate / "cort_si").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repo root containing the cort_si package.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from cort_si import (
    adaptive_source_selection,
    construct_Y,
    construct_Sigma,
    construct_folds,
    construct_test_statistic,
    gen_data,
    solve_cort_model,
)

RESULTS_DIR = REPO_ROOT / "run_experiment" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
p = 300
nS = 100
nT = 50
K = 2
lambda_sel = 0.25
lambda0 = 0.5
T = 3

num_runs = 1500
seed = 0

alpha = 0.05
true_beta_scale = 0.0
rho = 0.0
sigma_noise = 1.0
source_shift_sd = 0.3
covariate_shift = False

bonf_log10_family_size = p * np.log10(2.0)

print(f"num_runs = {num_runs}")
print(f"fixed alpha = {alpha}")
print(f"Null model true_beta scale = {true_beta_scale}")
print(f"K = {K}, p = {p}, nS = {nS}, nT = {nT}")
print(f"Bonferroni family size uses 2^p with p = {p} (about 10^{bonf_log10_family_size:.1f})")
print("Bonferroni adjustment is evaluated in log-space to avoid overflow.")

In [ ]:
def generate_null_instance(iteration_seed):
    return gen_data.generate_data(
        p=p,
        nS=nS,
        nT=nT,
        K=K,
        true_beta=true_beta_scale,
        rho=rho,
        sigma_noise=sigma_noise,
        source_shift_sd=source_shift_sd,
        covariate_shift=covariate_shift,
        seed=iteration_seed,
    )


def bonferroni_adjust_pvalue(naive_pvalue, num_features):
    if naive_pvalue is None or not np.isfinite(naive_pvalue):
        return np.nan
    if naive_pvalue <= 0.0:
        return 0.0

    # Avoid constructing 2**p directly when p is large.
    log_adjusted = np.log(naive_pvalue) + (num_features * np.log(2.0))
    if log_adjusted >= 0.0:
        return 1.0
    return float(np.exp(log_adjusted))


def compute_random_selected_pvalues(X0, Y0, XS_list, YS_list, SigmaS_list, Sigma0, lambdak_list, feature_seed):
    folds = construct_folds(X0.shape[0], T=T, shuffle=False)
    I_obs = adaptive_source_selection(X0, Y0, XS_list, YS_list, folds, lambda_sel, verbose=False)
    _, beta0_hat, _, _ = solve_cort_model(
        X0,
        Y0,
        XS_list,
        YS_list,
        I_obs,
        lambda0,
        lambdak_list,
        verbose=False,
        label="Observed CoRT fit",
    )
    M_obs = [idx for idx, value in enumerate(beta0_hat) if value != 0]
    if not M_obs:
        return None

    feature_rng = random.Random(feature_seed)
    feature_idx = feature_rng.choice(M_obs)

    Y_obs = construct_Y(YS_list, Y0)
    Sigma = construct_Sigma(SigmaS_list, Sigma0)
    X0M = X0[:, M_obs]
    etaj, etajTy = construct_test_statistic(feature_idx, X0M, Y_obs, M_obs, Y0.shape[0], Y_obs.shape[0])

    variance = float(np.asarray(etaj.T @ Sigma @ etaj).item())
    if not np.isfinite(variance) or variance <= 0.0:
        return None

    z_score = float(etajTy / np.sqrt(variance))
    naive_pvalue = float(2.0 * stats.norm.sf(abs(z_score)))
    bonf_pvalue = bonferroni_adjust_pvalue(naive_pvalue, X0.shape[1])

    return {
        "feature_idx": int(feature_idx),
        "naive_pvalue": naive_pvalue,
        "bonf_pvalue": bonf_pvalue,
        "selected_features": int(len(M_obs)),
        "selected_sources": int(len(I_obs)),
    }


def summarize_fpr_at_alpha(pvalues, alpha):
    if pvalues.size == 0:
        return np.array([], dtype=float), np.array([], dtype=float)

    rejection_indicators = (pvalues <= alpha).astype(float)
    running_fpr = np.cumsum(rejection_indicators) / np.arange(1, rejection_indicators.size + 1)
    return rejection_indicators, running_fpr


def run_fpr_experiment(num_runs, seed):
    naive_pvalues = []
    bonf_pvalues = []
    feature_indices = []
    selected_feature_counts = []
    selected_source_counts = []

    for iteration in range(num_runs):
        data_seed = seed + iteration
        XS_list, YS_list, X0, Y0, _, SigmaS_list, Sigma0, _ = generate_null_instance(data_seed)
        lambdak_list = [lambda0] * len(XS_list)
        pvalue_record = compute_random_selected_pvalues(
            X0,
            Y0,
            XS_list,
            YS_list,
            SigmaS_list,
            Sigma0,
            lambdak_list,
            feature_seed=data_seed,
        )

        if pvalue_record is not None:
            naive_pvalues.append(pvalue_record["naive_pvalue"])
            bonf_pvalues.append(pvalue_record["bonf_pvalue"])
            feature_indices.append(pvalue_record["feature_idx"])
            selected_feature_counts.append(pvalue_record["selected_features"])
            selected_source_counts.append(pvalue_record["selected_sources"])

        if (iteration + 1) % 10 == 0:
            print(f"Completed {iteration + 1}/{num_runs} runs")

    return {
        "naive_pvalues": np.asarray(naive_pvalues, dtype=float),
        "bonf_pvalues": np.asarray(bonf_pvalues, dtype=float),
        "feature_indices": np.asarray(feature_indices, dtype=int),
        "selected_feature_counts": np.asarray(selected_feature_counts, dtype=int),
        "selected_source_counts": np.asarray(selected_source_counts, dtype=int),
    }

In [ ]:
experiment_results = run_fpr_experiment(num_runs=num_runs, seed=seed)
naive_rejections, naive_running_fpr = summarize_fpr_at_alpha(experiment_results["naive_pvalues"], alpha)
bonf_rejections, bonf_running_fpr = summarize_fpr_at_alpha(experiment_results["bonf_pvalues"], alpha)

fpr_path = RESULTS_DIR / "fpr_naive_bonf_results.npz"
np.savez(
    fpr_path,
    naive_pvalues=experiment_results["naive_pvalues"],
    bonf_pvalues=experiment_results["bonf_pvalues"],
    feature_indices=experiment_results["feature_indices"],
    selected_feature_counts=experiment_results["selected_feature_counts"],
    selected_source_counts=experiment_results["selected_source_counts"],
    naive_rejections=naive_rejections,
    bonf_rejections=bonf_rejections,
    naive_running_fpr=naive_running_fpr,
    bonf_running_fpr=bonf_running_fpr,
    alpha=np.asarray([alpha], dtype=float),
    num_runs=np.asarray([num_runs], dtype=int),
    p=np.asarray([p], dtype=int),
 )

print(f"Saved naive/Bonferroni FPR results to {fpr_path}")
print(f"Valid null draws collected: {experiment_results['naive_pvalues'].size}/{num_runs}")
if naive_rejections.size > 0:
    print(f"Empirical naive FPR at alpha={alpha:.2f}: {naive_rejections.mean():.3f}")
if bonf_rejections.size > 0:
    print(f"Empirical Bonferroni FPR at alpha={alpha:.2f}: {bonf_rejections.mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if naive_running_fpr.size > 0:
    draw_index = np.arange(1, naive_running_fpr.size + 1)
    axes[0].plot(draw_index, naive_running_fpr, color="#aa2e25", linewidth=2, label="Naive running FPR")
    axes[0].plot(draw_index, bonf_running_fpr, color="#1f5673", linewidth=2, label="Bonferroni running FPR")
    axes[0].axhline(alpha, color="#35524a", linestyle="--", linewidth=2, label=f"Reference alpha = {alpha:.2f}")
    axes[0].legend()
else:
    axes[0].text(0.5, 0.5, "No valid null p-values", ha="center", va="center", transform=axes[0].transAxes)

axes[0].set_title(f"Running FPR at alpha={alpha:.2f}")
axes[0].set_xlabel("Number of valid null draws")
axes[0].set_ylabel("Cumulative rejection rate")
axes[0].set_xlim(1, max(1, naive_running_fpr.size))
axes[0].set_ylim(0.0, 1.0)
axes[0].grid(alpha=0.25)

axes[1].hist(experiment_results["naive_pvalues"], bins=np.linspace(0.0, 1.0, 11), density=True, color="#d8896b", edgecolor="#7a3b2e", alpha=0.85)
axes[1].axhline(1.0, color="#35524a", linestyle="--", linewidth=2, label="Uniform density = 1")
axes[1].set_title("Naive null p-values")
axes[1].set_xlabel("Naive p-value")
axes[1].set_ylabel("Density")
axes[1].set_xlim(0.0, 1.0)
axes[1].grid(alpha=0.25)
axes[1].legend()

axes[2].hist(experiment_results["bonf_pvalues"], bins=np.linspace(0.0, 1.0, 11), density=True, color="#9fc490", edgecolor="#35524a", alpha=0.85)
axes[2].set_title("Bonferroni-adjusted null p-values")
axes[2].set_xlabel("Bonferroni p-value")
axes[2].set_ylabel("Density")
axes[2].set_xlim(0.0, 1.0)
axes[2].grid(alpha=0.25)

plt.tight_layout()
plt.show()